# Econometría en Python

Cargar librerías necesarias:

In [ ]:
import numpy as np
from wooldridge import *
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats


Cargamos datos:

In [ ]:
dataWoo("hprice3", description=True)

In [ ]:
datos = dataWoo("hprice3")
y = datos["price"]
X = datos[["area", "cbd", "baths", "y81"]]


## Estadísticos de los Datos

In [ ]:
import matplotlib.pyplot as plt

media=np.mean(y)
Q1=np.quantile(y, 0.25)
Q3=np.quantile(y, 0.75)
Varianza=np.var(y)
DesviacionTipica=np.std(y)
Mediana=np.median(y)
histograma=plt.hist(y, bins='auto', rwidth=0.85)
plt.xlabel('y')
plt.ylabel('Frecuencia')
plt.title("Histrograma de y (salary)")
plt.show()
print(Q1, Mediana, Q3, DesviacionTipica, np.mean(y))

Calculamos modelo de Mínimos Cuadrados Ordinarios

In [ ]:
mco = smf.ols("price ~ area + cbd + baths + y81", data=datos).fit()
mco.summary()

## Extracción de Información

In [ ]:
# Suma de Cuadrados Explicada (SCE)

SCE = mco.ess

#Suma de Cuadrados de los Residuos

SCR = mco.ssr

# Suma de Cuadrados Total

SCT = mco.centered_tss

print("SCR: ", SCR, "SCE: ", SCE, "SCT: ", SCT)

## Normalidad de los Residuos

In [ ]:
import statsmodels.stats.api as sms
from statsmodels.compat import lzip
name = ["Jarque-Bera", "Chi^2 two-tail prob.", "Skew", "Kurtosis"]
test = sms.jarque_bera(mco.resid)
lzip(name, test)

## Multicolinealidad

In [ ]:
## Cálculo del FIV:
import statsmodels.stats.outliers_influence as oi

fivs=[oi.variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("FIVs: ", fivs)

### Cálculo del Número de condición

NC = np.sqrt(mco.condition_number)

### Matriz de Correlaciones:

corr_matrix=np.corrcoef(X.T)
print(corr_matrix)


In [ ]:
import statsmodels.graphics.api as smg
smg.plot_corr(corr_matrix, xnames=["area", "cbd", "baths", "y81"])
plt.show()

## Heteroscedasticidad

In [ ]:
#GOLDFELD-QUANDT (Muestras Pequeñas)
GQ=sms.het_goldfeldquandt(mco.model.endog, mco.model.exog, split=100)
print("GQ: ", GQ)

#BREUSH-PAGAN
BP=sms.het_breuschpagan(mco.resid, mco.model.exog)
print("BP: ", BP)

#WHITE
W=sms.het_white(mco.resid, mco.model.exog)

print("White: ", W)

In [ ]:
# Glejser:

z=datos["cbd"] 
for h in [-2,-1,-0.5,0.5, 1, 2]:
    # |e| = delta_0 + delta_1 z^h + eps
    mcoaux=sm.OLS(abs(mco.resid), sm.add_constant(z**h)).fit()
    pval=mcoaux.pvalues["cbd"]
    print("h: ", h, "-> pvalt:", pval, "R2: ", mcoaux.rsquared)
    
#Mínimos Cuadrados Ponderados
mcp = sm.WLS(y, sm.add_constant(X), weights=1./np.sqrt(z**2)).fit()    
mcp.summary()

## Autocorrelación

In [ ]:
### Durbin Watson:

from statsmodels.stats.stattools import durbin_watson
dw=durbin_watson(mco.resid)

print("DW: ", dw)


In [ ]:
## Corrección:

rho= 1 - dw/2

mco_autocorr=sm.GLSAR(y, sm.add_constant(X), rho=rho).iterative_fit(maxiter=1000)
mco_autocorr.summary()

# Cargar datos de ficheros:

In [ ]:
import pandas as pd #librería para manejo de datos

#### De Excel:
data= pd.read_excel ('stock.xlsx') #Lee parte de la base de datos "stocks"

In [ ]:
#### De CSV:
data= pd.read_csv('Theil.csv') #Lee parte de la base de datos "stocks"